# Temperature Agent — Agentic AI System
### Real-Time Urban Data Integration Platform
**Model:** `deepseek-r1:7b` via **Ollama** | **Data Source:** Open-Meteo API

## Step 1: Install Ollama on Google Colab
Ollama runs locally. We install it using a shell script inside Colab.

In [ ]:
# Install zstd
!sudo apt-get update
!sudo apt-get install -y zstd
!apt-get install -y pciutils lshw

# Then install Ollama again
!curl -fsSL https://ollama.com/install.sh | sh

## Step 2: Start Ollama Server in Background

In [ ]:
import subprocess
import time

# Start Ollama server in background
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give it a few seconds to start
time.sleep(5)
print("Ollama server started successfully.")

## Step 3: Pull DeepSeek-R1:7B Model
This downloads the model (~4.7 GB). Only needed once per Colab session.

In [ ]:
# Pull the deepseek-r1:7b model
!ollama pull deepseek-r1:7b

In [ ]:
# Verify model is available
!ollama list

## Step 4: Install Required Python Libraries

In [ ]:
!pip install requests ollama -q
print("Libraries installed.")

## Step 5: Define the Temperature Tool
This tool fetches **real-time temperature** from the **Open-Meteo API**.

It works in two steps:
- **Step A:** Geocode the city name → get latitude & longitude (via Open-Meteo Geocoding API)
- **Step B:** Fetch temperature using those coordinates (via Open-Meteo Weather API)

In [ ]:
import requests
import json

def get_temperature(city: str) -> dict:
    """
    Fetches real-time temperature for a given city using the Open-Meteo API.

    Args:
        city (str): Name of the city (e.g., 'New York', 'London', 'Tokyo')

    Returns:
        dict: Contains city, temperature, unit, and weather description
    """
    try:
        # --- Step A: Geocode city name to lat/lon ---
        geocode_url = "https://geocoding-api.open-meteo.com/v1/search"
        geo_params = {
            "name": city,
            "count": 1,
            "language": "en",
            "format": "json"
        }
        geo_response = requests.get(geocode_url, params=geo_params, timeout=10)
        geo_data = geo_response.json()

        if "results" not in geo_data or len(geo_data["results"]) == 0:
            return {"error": f"City '{city}' not found. Please check the city name."}

        location = geo_data["results"][0]
        lat = location["latitude"]
        lon = location["longitude"]
        country = location.get("country", "")
        resolved_name = location.get("name", city)

        # --- Step B: Fetch weather data using coordinates ---
        weather_url = "https://api.open-meteo.com/v1/forecast"
        weather_params = {
            "latitude": lat,
            "longitude": lon,
            "current_weather": True,
            "hourly": "temperature_2m,relativehumidity_2m,apparent_temperature",
            "forecast_days": 1
        }
        weather_response = requests.get(weather_url, params=weather_params, timeout=10)
        weather_data = weather_response.json()

        if "current_weather" not in weather_data:
            return {"error": "Weather data unavailable for this location."}

        current = weather_data["current_weather"]
        temp_c = current["temperature"]
        windspeed = current["windspeed"]
        weathercode = current["weathercode"]

        # WMO Weather Code descriptions (simplified)
        weather_descriptions = {
            0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
            45: "Foggy", 48: "Icy fog",
            51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
            61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
            71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
            80: "Slight showers", 81: "Moderate showers", 82: "Violent showers",
            95: "Thunderstorm", 96: "Thunderstorm with hail", 99: "Thunderstorm with heavy hail"
        }
        description = weather_descriptions.get(weathercode, "Unknown conditions")

        # Also fetch apparent (feels-like) temperature for current hour
        hourly = weather_data.get("hourly", {})
        apparent_temps = hourly.get("apparent_temperature", [])
        feels_like = apparent_temps[0] if apparent_temps else None

        return {
            "city": resolved_name,
            "country": country,
            "latitude": lat,
            "longitude": lon,
            "temperature_celsius": temp_c,
            "feels_like_celsius": feels_like,
            "wind_speed_kmh": windspeed,
            "weather_condition": description,
            "data_source": "Open-Meteo API (Real-Time)"
        }

    except requests.exceptions.Timeout:
        return {"error": "Request timed out. Please try again."}
    except requests.exceptions.ConnectionError:
        return {"error": "Network error. Check internet connection."}
    except Exception as e:
        return {"error": f"Unexpected error: {str(e)}"}


# Quick test of the tool
test_result = get_temperature("New York")
print(json.dumps(test_result, indent=2))


## Step 6: Define the Tool Schema (for DeepSeek Function Calling)
We define the tool in a structured JSON format so DeepSeek can decide when and how to call it.

In [ ]:
# Tool schema that tells the LLM about our temperature function
TEMPERATURE_TOOL_SCHEMA = {
    "name": "get_temperature",
    "description": (
        "Retrieves real-time temperature and current weather conditions for a specified city. "
        "Use this tool whenever the user asks about temperature, current weather, "
        "how hot or cold it is, or weather conditions in any city or location."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The name of the city to get temperature for (e.g., 'New York', 'London', 'Tokyo')"
            }
        },
        "required": ["city"]
    }
}

print("Tool schema defined.")
print(json.dumps(TEMPERATURE_TOOL_SCHEMA, indent=2))


## Step 7: Build the Temperature Agent
The agent:
1. Sends user query to DeepSeek-R1:7B with a system prompt explaining available tools
2. DeepSeek reasons and returns a tool call (city name to fetch)
3. Agent executes the tool and gets real data
4. Feeds data back to DeepSeek for a final natural language response

In [ ]:
import ollama
import re

# Available tools registry
AVAILABLE_TOOLS = {
    "get_temperature": get_temperature
}

# System prompt that instructs DeepSeek how to behave as an agent
SYSTEM_PROMPT = """You are a Temperature Agent — part of a Real-Time Urban Environmental Monitoring System.

Your ONLY job is to help users get real-time temperature and weather information for any city.

You have access to this tool:
- get_temperature(city: str): Fetches live temperature, feels-like temperature, wind speed, and weather conditions for a city.

INSTRUCTIONS:
1. When the user asks about temperature or weather in a city, extract the city name.
2. Respond with ONLY a JSON tool call in this exact format (nothing else):
   {"tool": "get_temperature", "parameters": {"city": "<city_name>"}}
3. Do NOT add any explanation before or after the JSON when making a tool call.
4. If the user's query is unclear or no city is mentioned, ask them to specify a city.
"""


def extract_tool_call(text: str) -> dict | None:
    """
    Extracts JSON tool call from DeepSeek's response.
    Handles cases where DeepSeek wraps response in <think> tags or adds extra text.
    """
    # Remove <think>...</think> blocks (DeepSeek-R1 reasoning traces)
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

    # Try to find JSON block
    json_pattern = r'\{[^{}]*"tool"[^{}]*\}'
    matches = re.findall(json_pattern, text, re.DOTALL)

    for match in matches:
        try:
            parsed = json.loads(match)
            if "tool" in parsed and "parameters" in parsed:
                return parsed
        except json.JSONDecodeError:
            continue

    # Try parsing the whole cleaned text as JSON
    try:
        parsed = json.loads(text)
        if "tool" in parsed:
            return parsed
    except json.JSONDecodeError:
        pass

    return None


def temperature_agent(user_query: str, verbose: bool = True) -> str:
    """
    Main Temperature Agent function.

    Args:
        user_query (str): Natural language query from the user
        verbose (bool): Print intermediate steps for transparency

    Returns:
        str: Final natural language response
    """
    print("-" * 60)
    print(f"User Query: {user_query}")
    print("-" * 60)

    # Send query to DeepSeek for reasoning
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]

    response = ollama.chat(
        model="deepseek-r1:7b",
        messages=messages
    )

    agent_response = response["message"]["content"]

    # Parse tool call from DeepSeek's response
    tool_call = extract_tool_call(agent_response)

    if not tool_call:
        # DeepSeek couldn't find a city — return its response directly
        clean_response = re.sub(r'<think>.*?</think>', '', agent_response, flags=re.DOTALL).strip()
        print(f"\nAgent Response: {clean_response}")
        return clean_response

    tool_name = tool_call.get("tool")
    tool_params = tool_call.get("parameters", {})

    # Execute the tool
    if tool_name not in AVAILABLE_TOOLS:
        return f"Error: Tool '{tool_name}' is not available."

    tool_result = AVAILABLE_TOOLS[tool_name](**tool_params)

    # Feed API result back to DeepSeek for final response
    if "error" in tool_result:
        final_prompt = f"The tool returned an error: {tool_result['error']}. Inform the user politely."
    else:
        final_prompt = f"""The user asked: "{user_query}"

Here is the real-time temperature data retrieved from the API:
{json.dumps(tool_result, indent=2)}

Respond using EXACTLY this format, with no extra text before or after:

Weather Report
--------------
City           : <city>, <country>
Temperature    : <temp_c>C / <temp_f>F
Feels Like     : <feels_like_c>C / <feels_like_f>F
Wind Speed     : <wind_speed> km/h
Conditions     : <weather_condition>
Data Source    : <data_source>

Final Response:
<A 2-3 sentence conversational summary mentioning the temperature in both Celsius and Fahrenheit, the feels-like temperature, current conditions, and a practical tip for the user.>

Do not include any JSON, markdown, bullet points, or additional sections beyond what is shown above."""

    final_messages = [
        {
            "role": "system",
            "content": "You are a weather reporting assistant. Output only the structured weather report followed by the Final Response section as specified. No extra text."
        },
        {
            "role": "user",
            "content": final_prompt
        }
    ]

    final_response = ollama.chat(
        model="deepseek-r1:7b",
        messages=final_messages
    )

    final_text = final_response["message"]["content"]
    # Remove reasoning traces from final output
    final_text = re.sub(r'<think>.*?</think>', '', final_text, flags=re.DOTALL).strip()

    print(f"\n{final_text}")
    print("-" * 60)

    return final_text


print("Temperature Agent defined and ready.")


## Step 8: Run the Temperature Agent — Test Queries
Let's test the agent with different natural language queries.

In [ ]:
# Test 1: Basic temperature query
result = temperature_agent("What is the current temperature in punjab?")

In [ ]:
# Test 2: Different phrasing
result = temperature_agent("How hot is it in jalandhar right now?")

In [ ]:
# Test 3: Another city
result = temperature_agent("Tell me the weather conditions in Tokyo today.")

In [ ]:
# Test 4: Informal query
result = temperature_agent("Is it cold in New York?")

## Step 9: Interactive Query Mode
Run this cell to interact with the agent continuously. Type `quit` to exit.

In [ ]:
print("Temperature Agent — Interactive Mode")
print("Type a city-related weather question. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["quit", "exit", "q"]:
        print("Goodbye!")
        break
    if not user_input:
        continue
    temperature_agent(user_input)
    print()
